In [ ]:
!pip install -q -U "transformers>=4.42.0" accelerate bitsandbytes "fastapi>=0.115.0,<0.124.0" uvicorn pyngrok python-multipart nest_asyncio

import os
import io
import torch
import uvicorn
import nest_asyncio
import traceback
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel
from contextlib import asynccontextmanager
from pyngrok import ngrok
from transformers import (
    AutoProcessor, 
    AutoModelForImageTextToText, 
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

HF_TOKEN = "" 
NGROK_TOKEN = ""
STATIC_DOMAIN = "choice-peacock-presently.ngrok-free.app"

DOCTOR_INSTRUCTIONS = """
You are **Dr. Mobile Doc**, a Senior Medical Consultant in Nigeria. 
You are chatting with a patient via a mobile app.

### **CRITICAL RULES (DO NOT BREAK):**
1.  **DIRECT SPEECH ONLY:** Do NOT output internal thoughts, lists of steps, or phrases like "(After receiving answer...)". Just talk to the patient.
2.  **NO PHYSICAL EXAMS:** You are on a phone. You cannot touch the patient. Do not say "I will check your vitals." Ask *them* to check.
3.  **KEEP IT SHORT:** Max 2-3 sentences. Chat style.

### **BEHAVIOR PROTOCOLS:**
1.  **CASUAL CHAT:**
    * If user says "Hi", "I'm fine", "And you?":
    * Reply warmly: "Sannu! I am doing well. Happy to hear you are fine." (Then stop. Do not force medical advice unless they ask).

2.  **MEDICAL MODE:**
    * If user mentions symptoms ("nausea", "pain", "fever"):
    * **NOW** use the [SYSTEM CONTEXT].
    * Example: "Sannu/Sorry about the nausea. Since your Widal test was positive for Salmonella, that is likely the cause. Are you vomiting too?"

3.  **TRIAGE:**
    * Chest Pain/Breathing Issues = "🚨 **EMERGENCY:** Go to the hospital immediately."
"""

MEDICAL_PROTOCOLS = {
    "fever": {
        "condition": "Febrile Illness",
        "suspects": "Malaria, Typhoid, Stress",
        "triage_questions": ["How high is the temperature?", "Bitter taste/Joint pains?", "Treated Malaria recently?"],
        "home_remedy": "Tepid sponging. Stay hydrated.",
        "advice": "Rest. Sponge body with tepid water. Drink ORS.",
        "lab_tests": "Malaria Parasite (MP), Widal Test"
    },
    "stomach": {
        "condition": "Gastrointestinal Distress",
        "suspects": "Ulcer, Food Poisoning, Gastritis",
        "triage_questions": ["Burning or cramping?", "Worse when hungry?", "Eaten spicy food?"],
        "home_remedy": "Avoid pepper/caffeine. Small frequent meals.",
        "advice": "Avoid spicy foods/pepper. Eat small frequent meals. Take Mist Mag.",
        "lab_tests": "Stool Microscopy, H. Pylori"
    },
    "cough": {
        "condition": "Respiratory Infection",
        "suspects": "Flu, Pneumonia, Tuberculosis",
        "triage_questions": ["Dry or wet cough?", "Any chest pain?", "How long has it lasted?"],
        "home_remedy": "Warm water with honey/lemon. Steam inhalation.",
        "advice": "Avoid drinking Cold Water.",
        "lab_tests": "Chest X-Ray, Sputum Test"
    }
}

def get_protocol(user_text: str):
    text = user_text.lower()
    if any(k in text for k in ["fever", "malaria", "hot"]): return MEDICAL_PROTOCOLS["fever"]
    if any(k in text for k in ["stomach", "pain", "ulcer"]): return MEDICAL_PROTOCOLS["stomach"]
    return None

vision_model = None
vision_processor = None
chat_model = None
chat_tokenizer = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global vision_model, vision_processor, chat_model, chat_tokenizer
    
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    
    # Shared 4-bit Config (Crucial for fitting both models)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print("\n⏳ LOADING VISION (Med-Gemma 4B)...")
    try:
        MODEL_ID = "google/medgemma-4b-it" 
        vision_processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
        vision_model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, 
            quantization_config=bnb_config, 
            device_map="cuda:0", 
            token=HF_TOKEN
        )
        print("✅ VISION READY")
    except Exception as e:
        print(f"❌ VISION LOAD ERROR: {e}")

    print("\n⏳ LOADING CHAT (Gemma 2 2B)...")
    try:
        chat_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it", token=HF_TOKEN)
        chat_tokenizer.pad_token = chat_tokenizer.eos_token 
        
        chat_model = AutoModelForCausalLM.from_pretrained(
            "google/gemma-2-2b-it",
            quantization_config=bnb_config,
            device_map="cuda:0",
            token=HF_TOKEN
        )
        print("✅ CHAT READY")
    except Exception as e:
        print(f"❌ CHAT LOAD ERROR: {e}")
    
    yield
    del vision_model, chat_model
    torch.cuda.empty_cache()

app = FastAPI(lifespan=lifespan)
nest_asyncio.apply()

class ChatReq(BaseModel):
    user_text: str

@app.post("/chat")
async def chat_endpoint(req: ChatReq):
    if not chat_model: return {"response": "System Error: Brain not loaded."}

    try:
        # 1. Check for specific medical keywords to trigger protocols
        protocol = get_protocol(req.user_text)
        
        # 2. Base System Prompt
        system_msg = DOCTOR_INSTRUCTIONS
        
        # 3. Add Specific Protocol Advice IF detected
        if protocol:
            system_msg += (
                f"\n\n### **MEDICAL PROTOCOL ACTIVATED:**\n"
                f"User seems to complain of: {protocol['condition']}\n"
                f"Please ASK these triage questions first: {', '.join(protocol['triage_questions'])}\n"
                f"Keep these treatments in mind (suggest only after triage): {protocol['home_remedy']}"
            )

        # 4. Construct Message
        # The req.user_text ALREADY includes the "[SYSTEM CONTEXT] ... [USER SAYS]" block from Frontend
        messages = [{"role": "user", "content": f"{system_msg}\n\n{req.user_text}"}]
        
        print(f"\n👁️ User Input: {req.user_text}")
        
        prompt_text = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = chat_tokenizer(prompt_text, return_tensors="pt", padding=True, truncation=True).to(chat_model.device)

        outputs = chat_model.generate(
            **inputs, 
            max_new_tokens=250, 
            do_sample=True, 
            temperature=0.7, # Slightly higher for more natural "Chat" feel
            pad_token_id=chat_tokenizer.eos_token_id
        )
        
        response = chat_tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        print(f"✅ AI REPLY: {response}")
        return {"response": response.strip()}

    except Exception as e:
        traceback.print_exc()
        return {"response": f"Chat Error: {str(e)}"}
        
@app.post("/analyze_image")
async def analyze_endpoint(file: UploadFile = File(...), description: str = Form(""), mode: str = Form("general")):
    print(f"\n👁️ RECEIVED IMAGE: Mode='{mode}'")
    if not vision_model: return {"response": "System Error: Eyes not loaded."}
    
    try:
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        image = image.resize((448, 448)) 

        raw_input = description
        food_name = raw_input.split('(')[0].strip() # "Jollof Rice"
        
        # We pass the FULL `raw_input` into the prompt so the AI sees the "(Patient Summary...)" part
        
        if mode == "diet":
            prompt_text = (
                f"The user describes this food as: {food_name}. "
                f"Context from App: {raw_input}. "  
                "Do not sound like an AI, Do not Hallucinate."
                "You are a professional and certified Nigerian Medical Nutritionist Analyze the diet: "                
                "1. **Identification & Macros:** What is this? (Carbs/Protein/Fat). Calories? "
                "2. **Health Check:** Look at the 'Context' above. Does the patient have Diabetes, Ulcer, or Pregnancy? Is this safe? "
                "3. **Meal Timing:** check 'Last Meal' in context. Is this good timing? "
                "4. **Verdict:** Suggest a portion size (e.g., '1 wrap')."
                "5. **Recommendation:** "
                "   - If SAFE: Suggest a portion size. "
                "   - If UNSAFE: Suggest a LIGHTER Nigerian alternative (e.g., 'Try Pap/Akamu', 'Pepper Soup', 'Moi-moi'). "
                "   - **IMPORTANT:** Do NOT recommend starvation. Always suggest hydration or a light meal."
            )
        elif mode == "lab":
            prompt_text = (
                f"Analyze this lab report. Context: {raw_input}. "
                "1. **Diagnosis:** Summarize abnormal results (e.g., 'Typhoid Positive'). "
                "2. **Next Step:** Suggest immediate medical action."
            )
        else:
            prompt_text = f"Describe medical findings in this image. Context: {raw_input}"

        print(f"🚀 PROMPT: {prompt_text}")

        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt_text}]}]
        inputs = vision_processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(vision_model.device)
        
        outputs = vision_model.generate(**inputs, max_new_tokens=500, min_new_tokens=20, do_sample=False)
        input_len = inputs["input_ids"].shape[-1]
        response = vision_processor.decode(outputs[0][input_len:], skip_special_tokens=True)

        print(f"✅ AI REPLY: {response}")
        return {"response": response.strip()}

    except Exception as e:
        print("\n❌ FATAL VISION ERROR:")
        traceback.print_exc()
        return {"response": f"Vision Error: {str(e)}"}

if __name__ == "__main__":
    if NGROK_TOKEN != "":
        print("⚠️ WARNING: You forgot to set your NGROK_TOKEN!")
    else:
        ngrok.set_auth_token(NGROK_TOKEN)
        try:
            public_url = ngrok.connect(8000, domain=STATIC_DOMAIN).public_url
            print(f"\n🚀 BACKEND LIVE (CHAT + VISION): {public_url}")
        except:
            print("⚠️ Static domain failed. Using random URL.")
            print(f"🚀 SERVER LIVE AT: {ngrok.connect(8000).public_url}")

    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



🚀 BACKEND LIVE (CHAT + VISION): https://choice-peacock-presently.ngrok-free.app


INFO:     Started server process [587]
INFO:     Waiting for application startup.



⏳ LOADING VISION (Med-Gemma 4B)...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

✅ VISION READY

⏳ LOADING CHAT (Gemma 2 2B)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ CHAT READY

👁️ User Input: [SYSTEM CONTEXT:       PATIENT SUMMARY:
      - Name: Isa Usman Musa
      - Phone: 08033617623
      - Age: 31
      - Vitals: 168cm, 70kg
      - Genotype: AA | Blood Group: B+
      - Allergies: None
      - Existing Conditions: None
      - Family History: Hypertension
       
 RECENT CONSULTATIONS: 2026-02-07: lab Analysis (Lab Report): Okay, I've analyzed the provided lab report. Here's a breakdown of the key findings and their implications:

**Widal Test Results:**

*   **S. Typhi O:** 1/80 (Negative)
*   **S. Paratyphi A:** 1/80 (Negative)
*   **S. Paratyphi B:** 1/80 (Negative)
*   **S. Paratyphi C:** 1/80 (Positive)

**Interpretation:**

The Widal test detects antibodies against *Salmonella* bacteria. A positive result indicates the presence of these antibodies, suggesting a past or current infection with *Salmonella* species. The fact that *S. Paratyphi C* is positive, while the other *Salmonella* serotypes are negative, strongly suggests an infe